<a href="https://colab.research.google.com/github/hugolo000/DigitalImage_PROJECT/blob/main/notebooks/01_data_extraction_and_eda.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import zipfile
import urllib.request
from pathlib import Path

# Step 1: Define dataset source URL and output directories
DATASET_URL = "https://github.com/alexgkendall/SegNet-Tutorial/archive/refs/heads/master.zip"
DATA_DIR = Path("./data")
RAW_ZIP_PATH = DATA_DIR / "camvid_raw.zip"
EXTRACTED_DIR = DATA_DIR / "extracted"

# Step 2: Download the zip archive if it does not exist locally
DATA_DIR.mkdir(parents=True, exist_ok=True)
if not RAW_ZIP_PATH.exists():
    print("[INFO] Downloading CamVid dataset...")
    urllib.request.urlretrieve(DATASET_URL, RAW_ZIP_PATH)
    print("[INFO] Download complete.")

# Step 3: Extract the dataset archive
if not EXTRACTED_DIR.exists():
    print("[INFO] Unzipping dataset files...")
    with zipfile.ZipFile(RAW_ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall(EXTRACTED_DIR)
    print("[INFO] Extraction complete.")

# Define base path to CamVid data folder
CAMVID_PATH = EXTRACTED_DIR / "SegNet-Tutorial-master" / "CamVid"
print(f"[INFO] CamVid root path: {CAMVID_PATH}")

[INFO] Downloading CamVid dataset...


2

In [ ]:
import pandas as pd

def load_camvid_split(txt_filename: str, base_path: Path) -> pd.DataFrame:
    """
    Parses a CamVid split file (e.g., train.txt) and returns a DataFrame
    with resolved absolute paths for images and masks.
    """
    txt_path = base_path / txt_filename
    image_paths = []
    mask_paths = []

    with open(txt_path, 'r') as file:
        for line in file:
            parts = line.strip().split()
            if len(parts) == 2:
                # Clean relative leading slashes
                img_rel = parts[0].lstrip('/')
                mask_rel = parts[1].lstrip('/')

                img_abs = (base_path / img_rel).resolve()
                mask_abs = (base_path / mask_rel).resolve()

                image_paths.append(str(img_abs))
                mask_paths.append(str(mask_abs))

    return pd.DataFrame({'image_path': image_paths, 'mask_path': mask_paths})

# Build DataFrames for each dataset split
df_train = load_camvid_split('train.txt', CAMVID_PATH)
df_val = load_camvid_split('val.txt', CAMVID_PATH)
df_test = load_camvid_split('test.txt', CAMVID_PATH)

print(f"Dataset Summary:")
print(f" - Training samples: {len(df_train)}")
print(f" - Validation samples: {len(df_val)}")
print(f" - Testing samples: {len(df_test)}")

3

In [ ]:
import cv2
import matplotlib.pyplot as plt

def display_sample_pairs(df: pd.DataFrame, num_samples: int = 3):
    """
    Displays side-by-side comparisons of raw driving images and ground truth masks.
    """
    fig, axes = plt.subplots(num_samples, 2, figsize=(12, 4 * num_samples))

    for i in range(num_samples):
        # Read BGR image and convert to RGB
        img_bgr = cv2.imread(df.iloc[i]['image_path'])
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

        # Read segmentation mask in grayscale
        mask = cv2.imread(df.iloc[i]['mask_path'], cv2.IMREAD_GRAYSCALE)

        # Plot RGB Image
        axes[i, 0].imshow(img_rgb)
        axes[i, 0].set_title(f"Sample {i+1}: Input Image ({img_rgb.shape[1]}x{img_rgb.shape[0]})")
        axes[i, 0].axis('off')

        # Plot Ground Truth Mask
        axes[i, 1].imshow(mask, cmap='tab20')
        axes[i, 1].set_title(f"Sample {i+1}: Ground Truth Mask")
        axes[i, 1].axis('off')

    plt.tight_layout()
    plt.show()

# Visualize 3 samples from the training set
display_sample_pairs(df_train, num_samples=3)